<a href="https://colab.research.google.com/github/Salalahmdi/Advanced-Statistics-Transportation/blob/main/Assignment_01_AADT_IRI_Correlation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 01 — Correlation Between AADT and IRI

**Advanced Statistics and Probability in Transportation**

**Research question:** Compute the overall Pearson correlation between AADT and IRI, then compute it separately by road class. If the overall correlation differs from the within-class relationships, what does that tell you?

This notebook uses the `pavement_transportation_master_dataset.csv` file. Run all cells from top to bottom in Google Colab.

## 1. Load the dataset

Upload `pavement_transportation_master_dataset.csv` to the Colab runtime when prompted. You can skip the upload if the file is already available in `/content/`.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
filename = "pavement_transportation_master_dataset.csv"

if not os.path.exists(filename) and not os.path.exists("/content/" + filename):
    from google.colab import files
    files.upload()

path = filename if os.path.exists(filename) else "/content/" + filename
df = pd.read_csv(path)
required = ["AADT_veh_day", "IRI_m_per_km", "road_class"]
missing = [col for col in required if col not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

print(f"Loaded {len(df):,} rows and {len(df.columns)} columns.")
display(df[required].head())

## 2. Calculate overall and within-class correlations

Pearson’s correlation coefficient ($r$) measures the **strength and direction of a linear association**. Its range is from −1 to +1. Correlation does not, by itself, establish causation.

In [ ]:
data = df[required].copy()
data["AADT_veh_day"] = pd.to_numeric(data["AADT_veh_day"], errors="coerce")
data["IRI_m_per_km"] = pd.to_numeric(data["IRI_m_per_km"], errors="coerce")
data = data.dropna(subset=required)

overall_r = data["AADT_veh_day"].corr(data["IRI_m_per_km"], method="pearson")

results = (data.groupby("road_class", sort=True)
           .apply(lambda group: pd.Series({
               "Observations": len(group),
               "Pearson r": group["AADT_veh_day"].corr(group["IRI_m_per_km"])
           }), include_groups=False)
           .reset_index())
results["Observations"] = results["Observations"].astype(int)

print(f"Overall correlation (n={len(data):,}): r = {overall_r:.3f}")
display(results.round(3))

## 3. Visualize the overall relationship

In [ ]:
plt.figure(figsize=(9, 6))
sns.regplot(data=data, x="AADT_veh_day", y="IRI_m_per_km",
            scatter_kws={"alpha": 0.25, "s": 15}, line_kws={"color": "black"})
plt.title(f"AADT vs. IRI — All Road Classes (Pearson r = {overall_r:.3f})")
plt.xlabel("Annual Average Daily Traffic (vehicles/day)")
plt.ylabel("International Roughness Index (m/km)")
plt.tight_layout()
plt.savefig("assignment01_overall_correlation.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Visualize the relationships by road class

Separate regression lines help show whether the association found in the pooled dataset is representative of each class.

In [ ]:
classes = sorted(data["road_class"].unique())
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
for ax, cls in zip(axes.flat, classes):
    group = data.loc[data["road_class"] == cls]
    r = group["AADT_veh_day"].corr(group["IRI_m_per_km"])
    sns.regplot(data=group, x="AADT_veh_day", y="IRI_m_per_km", ax=ax,
                scatter_kws={"alpha": 0.35, "s": 15},
                line_kws={"color": "black"})
    ax.set_title(f"{cls} (n={len(group)}, r={r:.3f})")
    ax.set_xlabel("AADT (vehicles/day)")
    ax.set_ylabel("IRI (m/km)")
for ax in axes.flat[len(classes):]:
    ax.set_visible(False)
fig.suptitle("AADT–IRI Correlation Within Each Road Class", fontsize=15)
plt.savefig("assignment01_by_road_class.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Interpretation and conclusion

Using the supplied dataset (**2,400 observations**), the overall Pearson correlation is approximately **+0.477**, indicating a moderate positive linear association between AADT and IRI. Within road classes, the correlations are **+0.490 (Expressway)**, **+0.170 (Arterial)**, **+0.164 (Collector)**, and **+0.085 (Local)**.

The pooled correlation is therefore not representative of most within-class relationships. Differences in typical traffic volumes and pavement roughness **between road classes** may contribute to the overall association. Road class is an important grouping variable and a potential confounder when interpreting the aggregate result.

This finding is consistent with an **aggregation effect**, but it is **not a strict reversal of direction** (the usual hallmark of Simpson’s paradox), because the within-class correlations remain positive. Moreover, correlation does **not** prove that greater traffic causes greater roughness. Additional controls and analysis would be needed for causal claims.

**Data source:** `pavement_transportation_master_dataset.csv` (provided course dataset).